# Appendix 02 — Streams & Async Overlap

> Independent appendix, distilled from *CUDA by Example* (Sanders & Kandrot),
> **Chapter 10 — Streams**. Informs the data-loading pipeline you meet in
> **Chapter 19 — The Full Training Loop**.

Until now the GPU did one thing at a time: copy data in, compute, copy results out — serially, with the GPU idle while PCIe transfers crawl. A modern GPU has **separate copy engines** that can move data *while* the SMs compute. **Streams** are how you ask for that overlap.

The payoff: if copying a batch and computing on it each take ~1 unit of time, doing them serially costs 2 units per batch; overlapping them costs ~1. For a training loop streaming batches off the host, that can nearly halve the wall-clock cost of data movement.

### Learning objectives

By the end you will:

- Explain **page-locked (pinned)** host memory (`cudaHostAlloc`) and why async copies require it.
- Use a **CUDA stream** + `cudaMemcpyAsync` to issue copy-compute-copy as an asynchronous pipeline.
- Overlap transfer with compute across **two streams** and measure the speedup.
- Connect this to prefetching training batches in `llm.c` (Chapter 19).


## 1. Concept — Pinned Memory & Async Copies

Ordinary host memory (`malloc`) is **pageable**: the OS may move or swap it. The GPU's DMA engine can't safely copy from memory that might move, so `cudaMemcpy` from pageable memory secretly stages through a pinned bounce buffer — and it's **synchronous** (blocks the CPU).

**Page-locked (pinned)** memory is allocated with `cudaHostAlloc` and never moves:

```c
float* h;  cudaHostAlloc(&h, bytes, cudaHostAllocDefault);   // pinned
...
cudaFreeHost(h);
```

Pinned memory unlocks two things:

1. **Higher copy bandwidth** — the DMA engine reads it directly, no bounce buffer.
2. **True async copies** — `cudaMemcpyAsync` returns immediately and runs on a copy engine, so it can overlap with kernels. (`cudaMemcpyAsync` from *pageable* memory silently falls back to synchronous — a common trap.)

Caveat: pinned memory is a scarce system resource (it can't be swapped). Allocating gigabytes of it can destabilize the machine. Pin the buffers you stream through, not everything.


## 2. Concept — Streams

A **stream** is an ordered queue of GPU operations. Work in *one* stream runs in issue order; work in *different* streams can run **concurrently** (hardware permitting). The default "null" stream (`0`) is special — it synchronizes with everything — so to get overlap you create explicit non-default streams:

```c
cudaStream_t s; cudaStreamCreate(&s);
cudaMemcpyAsync(d_a, h_a, bytes, cudaMemcpyHostToDevice, s);   // on stream s
kernel<<<grid, block, 0, s>>>(d_a, ...);                       // on stream s
cudaMemcpyAsync(h_r, d_r, bytes, cudaMemcpyDeviceToHost, s);
...
cudaStreamSynchronize(s);     // wait for stream s to drain
```

With **two** streams processing alternate chunks, the copy-in of chunk *k+1* (stream B's copy engine) overlaps the compute of chunk *k* (stream A's SMs) overlaps the copy-out of chunk *k-1*. The engines stay busy.


In [ ]:
!mkdir -p course/appendix02_build


In [ ]:
%%writefile course/appendix02_build/streams.cu
#include <stdio.h>
#include <math.h>
#include <cuda_runtime.h>

#define N        (1 << 22)     // total elements
#define CHUNK    (1 << 18)     // elements per chunk
#define NCHUNKS  (N / CHUNK)

// a deliberately compute-heavy kernel so compute time is comparable to copy time
__global__ void work(const float* a, const float* b, float* c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) {
        float x = a[i], y = b[i], acc = 0.0f;
        for (int k = 0; k < 64; k++) acc += sinf(x + k) * cosf(y - k);
        c[i] = acc;
    }
}

int main(void) {
    // pinned host buffers (required for real async copies)
    float *ha, *hb, *hc;
    cudaHostAlloc(&ha, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hb, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hc, N*sizeof(float), cudaHostAllocDefault);
    for (int i = 0; i < N; i++) { ha[i] = i*1e-4f; hb[i] = (N-i)*1e-4f; }

    int block = 256, grid = (CHUNK + block - 1) / block;
    cudaEvent_t t0, t1; cudaEventCreate(&t0); cudaEventCreate(&t1);

    // ===== Version 1: single stream, sequential per chunk =====
    cudaStream_t s; cudaStreamCreate(&s);
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, CHUNK*sizeof(float)); cudaMalloc(&d_b, CHUNK*sizeof(float)); cudaMalloc(&d_c, CHUNK*sizeof(float));
    cudaEventRecord(t0);
    for (int ci = 0; ci < NCHUNKS; ci++) {
        int off = ci * CHUNK;
        cudaMemcpyAsync(d_a, ha+off, CHUNK*sizeof(float), cudaMemcpyHostToDevice, s);
        cudaMemcpyAsync(d_b, hb+off, CHUNK*sizeof(float), cudaMemcpyHostToDevice, s);
        work<<<grid, block, 0, s>>>(d_a, d_b, d_c, CHUNK);
        cudaMemcpyAsync(hc+off, d_c, CHUNK*sizeof(float), cudaMemcpyDeviceToHost, s);
    }
    cudaStreamSynchronize(s);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float ms1; cudaEventElapsedTime(&ms1, t0, t1);
    double check1 = 0; for (int i = 0; i < N; i += N/8) check1 += hc[i];

    // ===== Version 2: two streams, overlapped =====
    cudaStream_t sa, sb; cudaStreamCreate(&sa); cudaStreamCreate(&sb);
    float *a0,*b0,*c0,*a1,*b1,*c1;
    cudaMalloc(&a0, CHUNK*sizeof(float)); cudaMalloc(&b0, CHUNK*sizeof(float)); cudaMalloc(&c0, CHUNK*sizeof(float));
    cudaMalloc(&a1, CHUNK*sizeof(float)); cudaMalloc(&b1, CHUNK*sizeof(float)); cudaMalloc(&c1, CHUNK*sizeof(float));
    cudaEventRecord(t0);
    for (int ci = 0; ci < NCHUNKS; ci += 2) {
        int o0 = ci * CHUNK, o1 = (ci+1) * CHUNK;
        // depth-first issue keeps both streams' copy + compute engines busy
        cudaMemcpyAsync(a0, ha+o0, CHUNK*sizeof(float), cudaMemcpyHostToDevice, sa);
        cudaMemcpyAsync(a1, ha+o1, CHUNK*sizeof(float), cudaMemcpyHostToDevice, sb);
        cudaMemcpyAsync(b0, hb+o0, CHUNK*sizeof(float), cudaMemcpyHostToDevice, sa);
        cudaMemcpyAsync(b1, hb+o1, CHUNK*sizeof(float), cudaMemcpyHostToDevice, sb);
        work<<<grid, block, 0, sa>>>(a0, b0, c0, CHUNK);
        work<<<grid, block, 0, sb>>>(a1, b1, c1, CHUNK);
        cudaMemcpyAsync(hc+o0, c0, CHUNK*sizeof(float), cudaMemcpyDeviceToHost, sa);
        cudaMemcpyAsync(hc+o1, c1, CHUNK*sizeof(float), cudaMemcpyDeviceToHost, sb);
    }
    cudaStreamSynchronize(sa); cudaStreamSynchronize(sb);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float ms2; cudaEventElapsedTime(&ms2, t0, t1);
    double check2 = 0; for (int i = 0; i < N; i += N/8) check2 += hc[i];

    printf("single stream : %6.3f ms\n", ms1);
    printf("two streams   : %6.3f ms\n", ms2);
    printf("overlap speedup: %.2fx   results %s\n", ms1/ms2,
           (fabs(check1 - check2) < 1e-2) ? "match" : "DIFFER");

    cudaFreeHost(ha); cudaFreeHost(hb); cudaFreeHost(hc);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix02_build/streams course/appendix02_build/streams.cu && ./course/appendix02_build/streams


Both versions compute the same results; the two-stream version is much faster. On this RTX 4080 SUPER the gap is large (≈10× in the run above), and it's worth being precise about *why*, because two distinct effects stack:

1. **Copy/compute overlap.** Pure overlap of transfers with compute can hide at most the *smaller* of (copy time, compute time) — a ceiling of ~2× when the two are balanced.
2. **Concurrent kernels.** With two streams and separate buffers, the GPU also runs *two* `work` kernels at once. Each chunk's kernel launches only ~1024 blocks, far from filling the 80 SMs, so two co-resident kernels roughly double compute throughput on top of the overlap.

There's also a third, more mundane reason the ratio looks so dramatic: the single-stream baseline **reuses one set of device buffers**, so each chunk's input copy must wait for the previous chunk's kernel and output copy to finish (a write-after-read dependency). That makes the baseline fully serial — a deliberately pessimistic reference that makes the win vivid. The honest takeaway: **streams buy you both overlap and concurrency; the exact multiple depends on how starved the single-stream version was and how far from full the kernels are.** Don't memorize "10×" or "2×" — measure.


## 3. Translation Bridge — Overlap in `llm.c`

| Book (Ch10) | `llm.c` / training | Role |
|---|---|---|
| `cudaHostAlloc` pinned buffers | pinned staging for token batches | enables true async H2D |
| `cudaMemcpyAsync` on a stream | prefetch the *next* batch while the GPU trains on the current one | hide data-loading latency |
| two streams overlapping copy+compute | compute stream vs. copy/comm stream | keep SMs fed |
| `cudaStreamSynchronize` | barrier before stepping the optimizer | correctness |

The same principle scales up in multi-GPU training (Chapter 20): NCCL all-reduce of gradients runs on a **separate stream**, overlapping with the backward pass's compute, so communication is (partly) free. "Overlap copy/communication with compute" is one of the highest-leverage performance ideas in the whole course.


## 4. Common Pitfalls

- **`cudaMemcpyAsync` from pageable memory is secretly synchronous.** No pinned buffer → no overlap, silently. Always `cudaHostAlloc` the host side.
- **The default stream (0) serializes.** Launch overlapping work on *explicit* non-default streams.
- **Over-pinning** — pinned memory can't be swapped; pinning too much starves the OS. Pin only your streaming buffers.
- **Reusing a device buffer across overlapping streams** creates a data race — each stream needs its own (`a0/a1`, `c0/c1` above).
- **Forgetting to synchronize** the stream before reading host results → you read stale data.


## 5. TODO Exercise — Make the Copy Async

The loop below uses a plain synchronous `cudaMemcpy` on the default stream, killing any overlap. Convert the three transfers to `cudaMemcpyAsync` on stream `s`. Fill in the TODOs.


In [ ]:
%%writefile course/appendix02_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N       (1 << 20)
#define CHUNK   (1 << 17)

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main(void) {
    float *ha, *hb, *hc;
    cudaHostAlloc(&ha, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hb, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hc, N*sizeof(float), cudaHostAllocDefault);
    for (int i = 0; i < N; i++) { ha[i] = i; hb[i] = 2*i; }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, CHUNK*4); cudaMalloc(&d_b, CHUNK*4); cudaMalloc(&d_c, CHUNK*4);
    cudaStream_t s; cudaStreamCreate(&s);
    int block = 256, grid = (CHUNK+block-1)/block;

    for (int off = 0; off < N; off += CHUNK) {
        // TODO 1: async copy ha+off -> d_a (CHUNK floats) on stream s
        cudaMemcpy(d_a, ha+off, CHUNK*4, cudaMemcpyHostToDevice);
        // TODO 2: async copy hb+off -> d_b on stream s
        cudaMemcpy(d_b, hb+off, CHUNK*4, cudaMemcpyHostToDevice);
        add<<<grid, block, 0, s>>>(d_a, d_b, d_c, CHUNK);
        // TODO 3: async copy d_c -> hc+off on stream s
        cudaMemcpy(hc+off, d_c, CHUNK*4, cudaMemcpyDeviceToHost);
    }
    cudaStreamSynchronize(s);

    int ok = 1; for (int i = 0; i < N; i++) if (hc[i] != 3.0f*i) { ok = 0; break; }
    printf("%s  (hc[0..3] = %.0f %.0f %.0f %.0f)\n", ok ? "PASS" : "FAIL", hc[0], hc[1], hc[2], hc[3]);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix02_build/exercise1 course/appendix02_build/exercise1.cu && ./course/appendix02_build/exercise1


### Solution

In [ ]:
%%writefile course/appendix02_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N       (1 << 20)
#define CHUNK   (1 << 17)

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main(void) {
    float *ha, *hb, *hc;
    cudaHostAlloc(&ha, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hb, N*sizeof(float), cudaHostAllocDefault);
    cudaHostAlloc(&hc, N*sizeof(float), cudaHostAllocDefault);
    for (int i = 0; i < N; i++) { ha[i] = i; hb[i] = 2*i; }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, CHUNK*4); cudaMalloc(&d_b, CHUNK*4); cudaMalloc(&d_c, CHUNK*4);
    cudaStream_t s; cudaStreamCreate(&s);
    int block = 256, grid = (CHUNK+block-1)/block;

    for (int off = 0; off < N; off += CHUNK) {
        cudaMemcpyAsync(d_a, ha+off, CHUNK*4, cudaMemcpyHostToDevice, s);   // TODO 1
        cudaMemcpyAsync(d_b, hb+off, CHUNK*4, cudaMemcpyHostToDevice, s);   // TODO 2
        add<<<grid, block, 0, s>>>(d_a, d_b, d_c, CHUNK);
        cudaMemcpyAsync(hc+off, d_c, CHUNK*4, cudaMemcpyDeviceToHost, s);   // TODO 3
    }
    cudaStreamSynchronize(s);

    int ok = 1; for (int i = 0; i < N; i++) if (hc[i] != 3.0f*i) { ok = 0; break; }
    printf("%s  (hc[0..3] = %.0f %.0f %.0f %.0f)\n", ok ? "PASS" : "FAIL", hc[0], hc[1], hc[2], hc[3]);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix02_build/exercise1_sol course/appendix02_build/exercise1_sol.cu && ./course/appendix02_build/exercise1_sol


## Recap

- **Pinned memory** (`cudaHostAlloc`) doesn't move, so the DMA engine can copy it directly and `cudaMemcpyAsync` is truly asynchronous. Async copy from pageable memory silently falls back to synchronous.
- **Streams** are ordered queues; work in different (non-default) streams runs concurrently. Two streams let copy and compute overlap.
- Overlap hides at most the smaller of (copy time, compute time); the ideal is ~2× when they're balanced.
- `llm.c` uses the same idea to prefetch batches and to overlap NCCL gradient all-reduce with the backward pass (Chapters 19–20).

### What's next

**Chapter 20a — Zero-Copy & Multi-GPU** (book Ch11): mapped/portable pinned memory the GPU reads directly over PCIe, and splitting work across multiple GPUs — the foundation under Chapter 20's NCCL/ZeRO data-parallel training. (Single-GPU box: the zero-copy parts run here; the true multi-GPU split is shown as code + expected output.)
